# b_Build_Model Runner

각 피험자 `Model_osim/` 폴더에 들어온 AddBiomech 산출물을 다음 파이프라인으로 처리:

1. **rename_AddBiomech_model** — `match_markers_but_ignore_physics.osim` → `SUB{num}_Scaled.osim`
2. **add_hand_mass_model**     — `SUB{num}_Scaled_HeavyHand_{w}kg.osim` 생성 (각 손에 w/2 kg)
3. **add_box_weldjoint_model** — `SUB{num}_Scaled_{WeldBox|SplitBox}_{w}kg.osim` 생성 (ADDBOX 활용)

피험자별 가능한 박스 무게(w_kg)는 `SUB_Info.subjects[namecode]['conditions']` 키의 prefix
(`7kg`, `10kg`, `15kg`)에서 자동 추출됩니다.

In [ ]:
import os
import sys

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, 'Codes')):  # 노트북 실행 위치가 repo root 가 아닌 경우
    repo_root = r'C:/Users/ok/Documents/GitHub/BOX'      # TODO: change to your path

work_dir = os.path.join(repo_root, 'Codes', 'b_Build_Model')
codes_dir = os.path.dirname(work_dir)
os.chdir(work_dir)
for p in (work_dir, codes_dir):
    if p not in sys.path:
        sys.path.insert(0, p)

print('cwd      :', os.getcwd())
print('work_dir :', work_dir)

In [ ]:
from SUB_Info import subjects

print('Available subjects:', list(subjects.keys()))

namecode = '260306_KTY'  # TODO: 필요 시 변경
info = subjects[namecode]
print('sex       :', info['sex'])
print('protocol  :', info['protocol'])
print('conditions:', list(info['conditions'].keys()))

## 1) rename_AddBiomech_model

`match_markers_but_ignore_physics.osim` → `SUB{num}_Scaled.osim`

In [ ]:
from rename_AddBiomech_model import rename_scaled_model

scaled_path = rename_scaled_model(namecode, overwrite=False)
print('Scaled model:', scaled_path)

## 2) add_hand_mass_model

7kg → 각 손 +3.5kg, 10kg → +5kg, 15kg → +7.5kg

In [ ]:
from add_hand_mass_model import build_heavyhand_models

heavyhand_outputs = build_heavyhand_models(namecode, overwrite=True)
for p in heavyhand_outputs:
    print(' -', p)

## 3) add_box_weldjoint_model

`ADDBOXtoOSIM(..., Constraint=True)`  → `WeldBox`
`ADDBOXtoOSIM(..., Constraint=False)` → `SplitBox`

In [ ]:
from add_box_weldjoint_model import build_box_models

box_outputs = build_box_models(namecode, overwrite=True)
for p in box_outputs:
    print(' -', p)

## (옵션) 전체 피험자 일괄 실행

In [ ]:
from rename_AddBiomech_model import rename_all
from add_hand_mass_model import build_all as build_all_heavyhand
from add_box_weldjoint_model import build_all as build_all_box

rename_all(overwrite=False)
build_all_heavyhand(overwrite=True)
build_all_box(overwrite=True)